# Environmental inputs for wildfire forecasting in central Spain, 2012–2025

This notebook prepares environmental data for the three-day wildfire forecast on the existing
grid of 288 cells at 0.25° resolution. It includes every UTC issue date from January 1, 2012,
through December 31, 2025. Earth Engine aggregates the source data to the grid. Google Drive
stores the completed tables and the downloaded blocks.

| Input | Source and coverage | Output |
|---|---|---|
| Day 1 weather | [NOAA GFS](https://developers.google.com/earth-engine/datasets/catalog/NOAA_GFS0P25), eligible issues from July 2, 2015 | `weather_features.parquet` |
| Days 2 and 3 weather | Same GFS run as day 1. Eligible issues from July 10, 2017, when the archive began to support the full horizons | `gfs_days_2_3.parquet` |
| Mean elevation and slope | [NASADEM](https://developers.google.com/earth-engine/datasets/catalog/NASA_NASADEM_HGT_001), one static layer | `elevation_by_cell.parquet` |
| Land cover fractions | [CORINE](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_CORINE_V20_100m), 2012 and 2018 snapshots | `corine_2012_by_cell.parquet`, `corine_2018_by_cell.parquet` |
| Surface soil moisture | [SMAP L4 V008](https://developers.google.com/earth-engine/datasets/catalog/NASA_SMAP_SPL4SMGP_008), observations from March 31, 2015, plus the original publication lag and trailing window | `smap_v008_lag7_features.parquet` |
| Vegetation moisture proxy (NDMI) | [MODIS Terra MOD09A1.061](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD09A1), available throughout 2012–2025 where quality checks pass | `ndmi_features.parquet` |
| Fire Weather Index | Calculated from GFS with xclim after the 120-day initialization period | `fwi_features.parquet` |

The two GFS weather tables and the recalculated FWI table are stored in the
`gfs_physical_qc/` subfolder. Earlier cached forecast blocks are reused when their
values cannot have been changed by the previous upper limits or selective masking.
Ambiguous blocks are fetched again. The existing cached weather inputs for FWI and
the other source caches are reused.

**Before running:** Use a CPU runtime. Set `EE_PROJECT` below to a Google Cloud project
registered for Earth Engine or add it as a Colab secret. After setup, run the Earth Engine
authentication cell. Open the Google authorization link, select the authentication project,
click **Generate Token**, and paste the verification code into the Colab prompt. A connection
test result of `1` means the notebook can access Earth Engine. Continue with the data cells.

If the runtime restarts, rerun the cells to reuse completed tables and blocks. All outputs
are stored in `MyDrive/spain_firms_2012_2025/features/`.

The final cell writes `environmental_features.parquet` with one row per issue and cell.
It keeps unavailable measurements as nulls and reports actual coverage by year in
`environmental_feature_coverage.csv`. The combined table retains the original convention
of using CORINE 2018 as a fixed land cover layer, including for earlier dates. CORINE 2012
is written as a separate snapshot. Use the keys `issue_utc`, `cell_y`, and `cell_x` to join
these features with a separate fire training table.

In [ ]:
# 1. Set up Google Drive and the forecast grid.
from functools import lru_cache
from hashlib import sha256
from importlib.util import find_spec
from pathlib import Path
from time import sleep
from uuid import uuid4
import json
import subprocess
import sys

import numpy as np
import pandas as pd
from google.colab import drive, userdata
from IPython.display import display

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/spain_firms_2012_2025")
OUTPUT_ROOT = DRIVE_ROOT / "features"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EE_PROJECT = ""  # Optional if the Colab secret EE_PROJECT is set.
START_UTC = pd.Timestamp("2012-01-01", tz="UTC")
STOP_UTC = pd.Timestamp("2026-01-01", tz="UTC")  # Exclusive.

if find_spec("pyarrow") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])

BBOX = (-6.0, 38.0, -1.5, 42.0)  # west, south, east, north
CELL_DEG = 0.25
west, south, east, north = BBOX
if (not (west < east and south < north and CELL_DEG > 0)
    or not all(np.isclose(edge / CELL_DEG, round(edge / CELL_DEG)) for edge in BBOX)):
    raise ValueError("The bounding box must align with the configured grid.")
ys = np.arange(round(south / CELL_DEG), round(north / CELL_DEG))
xs = np.arange(round(west / CELL_DEG), round(east / CELL_DEG))
CELL_KEYS = ["cell_y", "cell_x"]
KEYS = ["issue_utc", *CELL_KEYS]

GFS_DATASET = "NOAA/GFS0P25"
RUN_AGE_HOURS = 6
GFS_ARCHIVE_START = pd.Timestamp("2015-07-01", tz="UTC")
GFS_DAY1_FIRST_ISSUE = (GFS_ARCHIVE_START + pd.Timedelta(hours=RUN_AGE_HOURS)).ceil("D")
# A full days 2–3 forecast requires leads beyond 36 hours.
GFS_EXTENDED_RUN_START = pd.Timestamp("2017-07-09", tz="UTC")
GFS_DAY23_FIRST_ISSUE = (GFS_EXTENDED_RUN_START + pd.Timedelta(hours=RUN_AGE_HOURS)).ceil("D")
BLOCK_DAYS = 7
GFS_FEATURES = ["temp_max_24h", "rh_min_24h", "wind_max_24h", "precip_sum_24h"]
GFS_COLUMNS = KEYS + ["gfs_run_utc"] + GFS_FEATURES
MISSING_PIXEL = -9999.0
# Earth Engine returns pixel rows from north to south.
transform = [CELL_DEG, 0, west, 0, -CELL_DEG, north]
pixel_grid = {
    "dimensions": {"width": len(xs), "height": len(ys)},
    "affineTransform": dict(zip(
        ["scaleX", "shearX", "translateX", "shearY", "scaleY", "translateY"], transform,
    )),
    "crsCode": "EPSG:4326",
}
grid_id = sha256(repr((GFS_DATASET, BBOX, CELL_DEG, RUN_AGE_HOURS, BLOCK_DAYS, "v1")).encode()).hexdigest()[:12]
GFS_QC_ROOT = OUTPUT_ROOT / "gfs_physical_qc"
legacy_cache_dir = OUTPUT_ROOT / "gfs_cache" / grid_id
cache_dir = GFS_QC_ROOT / "gfs_cache" / grid_id
weather_path = GFS_QC_ROOT / "weather_features.parquet"

DEM_DATASET = "NASA/NASADEM_HGT/001"
CLC_YEARS = (2012, 2018)
CLC_YEAR = 2018  # Retain the original static layer in the combined feature table.
CORINE_COLLECTION = "COPERNICUS/CORINE/V20/100m"
TERRAIN_FEATURES = ["elevation_mean_m", "slope_mean_deg"]
COVER_GROUPS = {
    "forest_frac": [311, 312, 313],
    "shrubland_frac": [322, 323, 324],
    "natural_grassland_frac": [321],
    "agriculture_frac": [211, 212, 213, 221, 222, 223, 231, 241, 242, 243, 244],
}
COVER_FEATURES = list(COVER_GROUPS)

SMAP_DATASET = "NASA/SMAP/SPL4SMGP/008"
SMAP_ARCHIVE_START = pd.Timestamp("2015-03-31", tz="UTC")
SMAP_PUBLICATION_LAG_DAYS = 7
SMAP_HISTORY_DAYS = 7
SMAP_BLOCK_DAYS = 28
SMAP_FEATURES = ["smap_surface_daily", "smap_surface_mean_7d"]
SMAP_SOURCE_DAY = "smap_observation_day_utc"

NDMI_DATASET = "MODIS/061/MOD09A1"
NDMI_COMPOSITE_DAYS = 8
NDMI_PUBLICATION_BUFFER_DAYS = 3
NDMI_MAX_FALLBACK_DAYS = 24
NDMI_MIN_VALID_FRACTION = 0.20
NDMI_BLOCK_SIZE = 6
VEGETATION_FEATURES = ["ndmi_mean"]
NDMI_SOURCE_DATE = "ndmi_composite_start_utc"
NDMI_AVAILABLE_DATE = "ndmi_assumed_available_utc"
NDMI_COVERAGE = "ndmi_valid_fraction"

FWI_WEATHER = ["fwi_temp_c", "fwi_rh_pct", "fwi_wind_kmh", "fwi_rain_mm"]
FWI_GFS_BANDS = [
    "temperature_2m_above_ground", "relative_humidity_2m_above_ground",
    "u_component_of_wind_10m_above_ground", "v_component_of_wind_10m_above_ground",
]
FWI_GFS_RAIN = "total_precipitation_surface"
FWI_WARMUP_DAYS = 120
FWI_FIRST_ISSUE = GFS_DAY1_FIRST_ISSUE + pd.Timedelta(days=FWI_WARMUP_DAYS)
FWI_BLOCK_DAYS = 14

issue_days = pd.date_range(START_UTC, STOP_UTC, freq="D", inclusive="left").as_unit("ns")
grid_table = pd.MultiIndex.from_product([issue_days, ys, xs], names=KEYS).to_frame(index=False)
grid_table["year"] = grid_table["issue_utc"].dt.year
grid_cells = grid_table[CELL_KEYS].drop_duplicates(ignore_index=True)
print(f"Forecast grid: {len(grid_cells)} cells x {len(issue_days)} dates = {len(grid_table):,} rows.")
print(f"Output directory: {OUTPUT_ROOT}")

@lru_cache(maxsize=1)
def open_earth_engine():
    project = EE_PROJECT.strip()
    if not project:
        try:
            project = (userdata.get("EE_PROJECT") or "").strip()
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            pass
    if not project:
        raise ValueError("Set EE_PROJECT in this cell or in Colab Secrets to a Cloud project registered for Earth Engine.")
    if find_spec("ee") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "earthengine-api"])
    import ee

    ee.Authenticate(auth_mode="notebook", force=True)
    ee.Initialize(project=project)
    return ee

In [ ]:
# Authorize Earth Engine with a Google verification code and test the connection.
open_earth_engine.cache_clear()
earth_engine = open_earth_engine()
print("Earth Engine connection test:", earth_engine.Number(1).getInfo())


## Weather forecasts

All three daily horizons use the previous day's GFS run at 18:00 UTC. The features are
maximum temperature, minimum relative humidity, maximum wind speed, and total precipitation
for each UTC day. Downloads begin at the first eligible issue for each horizon. The earliest
archive runs extend to 36 hours, so the days 2–3 table starts later than the day 1 table.

Negative relative humidity, wind speed and precipitation are clipped to zero.
Negative Celsius temperatures are retained. Nonfinite values and temperatures below
absolute zero are marked missing. The log reports corrections by field and date.
Temperature, wind and precipitation have no upper clipping limits. Forecast
humidity above 100% is retained in the weather tables. Grid keys and issue/run
timestamps still require an exact match.

In [ ]:
# 2. Download GFS features for day 1.
def day_image(ee, issue):
    run = issue - pd.Timedelta(hours=RUN_AGE_HOURS)
    run_ms = run.value // 1_000_000
    weather = ee.ImageCollection(GFS_DATASET).filter(ee.Filter.eq("creation_time", run_ms))
    state_bands = [
        "temperature_2m_above_ground", "relative_humidity_2m_above_ground",
        "u_component_of_wind_10m_above_ground", "v_component_of_wind_10m_above_ground",
    ]
    rain_band = "total_precipitation_surface"
    state_names = ["temperature", "humidity", "u", "v"]
    state_leads = list(range(RUN_AGE_HOURS, RUN_AGE_HOURS + 24, 3))
    # Each lead ends a distinct six-hour accumulation.
    rain_leads = list(range(RUN_AGE_HOURS + 6, RUN_AGE_HOURS + 25, 6))

    def on_grid(image):
        return image.resample("bilinear").reproject(crs="EPSG:4326", crsTransform=transform)

    def select_step(lead, bands, names):
        valid_ms = (run + pd.Timedelta(hours=lead)).value // 1_000_000
        matching = weather.filter(ee.Filter.eq("forecast_hours", lead)).filter(
            ee.Filter.eq("forecast_time", valid_ms)
        )
        empty = ee.Image.constant([0] * len(names)).rename(names).updateMask(ee.Image.constant(0))
        return ee.Image(ee.Algorithms.If(
            matching.size().eq(1), on_grid(ee.Image(matching.first()).select(bands, names)), empty
        ))

    def state_image(lead):
        image = select_step(lead, state_bands, state_names)
        wind = image.select("u").pow(2).add(image.select("v").pow(2)).sqrt().rename("wind")
        return image.select(["temperature", "humidity"]).addBands(wind)

    def complete_reduce(collection, band, reducer, n, name):
        selected = collection.select(band)
        result = getattr(selected, reducer)().updateMask(selected.count().eq(n))
        return result.rename(name)

    state = ee.ImageCollection.fromImages([state_image(lead) for lead in state_leads])
    rain = ee.ImageCollection.fromImages([select_step(lead, [rain_band], ["rain"]) for lead in rain_leads])
    return complete_reduce(state, "temperature", "max", len(state_leads), GFS_FEATURES[0]).addBands([
        complete_reduce(state, "humidity", "min", len(state_leads), GFS_FEATURES[1]),
        complete_reduce(state, "wind", "max", len(state_leads), GFS_FEATURES[2]),
        complete_reduce(rain, "rain", "sum", len(rain_leads), GFS_FEATURES[3]),
    ]).toFloat().unmask(MISSING_PIXEL).set("system:index", issue.strftime("d%Y%m%d"))


def pixels_to_weather(pixels, block):
    if pixels.shape != (len(ys), len(xs)):
        raise ValueError(f"Expected grid {(len(ys), len(xs))}, got {pixels.shape}.")
    grid_y, grid_x = np.meshgrid(ys[::-1], xs, indexing="ij")
    parts = []
    for issue in block:
        data = {"issue_utc": issue, "cell_y": grid_y.ravel(), "cell_x": grid_x.ravel(),
                "gfs_run_utc": issue - pd.Timedelta(hours=RUN_AGE_HOURS)}
        for name in GFS_FEATURES:
            band = f"{issue.strftime('d%Y%m%d')}_{name}"
            values = pixels[band].astype(np.float32).ravel()
            data[name] = np.where(values == MISSING_PIXEL, np.nan, values)
        parts.append(pd.DataFrame(data))
    return pd.concat(parts, ignore_index=True)


def clean_gfs_values(frame, names):
    temperature = names[0]
    for name in names:
        values = pd.to_numeric(frame[name], errors="raise")
        invalid = np.isinf(values)
        if name == temperature:
            invalid |= values.lt(-273.15)
            negative = pd.Series(False, index=values.index)
        else:
            negative = values.lt(0) & ~invalid

        flagged = invalid | negative
        if flagged.any():
            dates = frame.loc[flagged, "issue_utc"]
            original = values[flagged]
            print(
                f"GFS quality {name} {dates.min():%Y-%m-%d} to {dates.max():%Y-%m-%d}: "
                f"{int(negative.sum())} clipped to zero, {int(invalid.sum())} set to NaN; "
                f"original values {original.min():.6g} to {original.max():.6g}.",
                flush=True,
            )
        frame[name] = values.mask(invalid).mask(negative, 0)
    return frame


def read_legacy_gfs_block(path, feature_groups):
    """Reuse a block only if its values are compatible with the new policy."""
    if not path.is_file():
        return None
    frame = pd.read_parquet(path)
    for names in feature_groups:
        _, humidity, _, precipitation = names
        # Earlier versions masked outliers and clipped some values to these upper limits.
        # Lower zeros from the old cache are compatible with clipping negatives to zero.
        if (frame[names].isna().any(axis=None) or frame[humidity].eq(100).any()
                or frame[precipitation].eq(1500).any()):
            return None
    return frame


def check_weather(frame, expected_keys):
    frame = frame[GFS_COLUMNS].copy()
    for name in ["issue_utc", "gfs_run_utc"]:
        frame[name] = pd.to_datetime(frame[name], utc=True).astype("datetime64[ns, UTC]")
    if not frame["gfs_run_utc"].eq(frame["issue_utc"] - pd.Timedelta(hours=RUN_AGE_HOURS)).all():
        raise ValueError("GFS run times do not match the configured lead time.")
    if frame.duplicated(KEYS).any() or len(frame) != len(expected_keys):
        raise ValueError("GFS has duplicated or incomplete grid keys.")
    frame = expected_keys.merge(frame, on=KEYS, how="left", validate="one_to_one")
    if frame["gfs_run_utc"].isna().any():
        raise ValueError("GFS is missing keys for the forecast grid.")
    return clean_gfs_values(frame, GFS_FEATURES)


def atomic_parquet(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.{uuid4().hex}.tmp")
    try:
        frame.to_parquet(tmp, index=False)
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)


def fetch_block(ee, block, expected_keys):
    stack = ee.ImageCollection.fromImages([day_image(ee, day) for day in block]).toBands()
    request = {"expression": stack, "fileFormat": "NUMPY_NDARRAY", "grid": pixel_grid}
    for attempt in range(3):
        try:
            pixels = ee.data.computePixels(request)
            break
        except ee.EEException as exc:
            retryable = any(token in str(exc).lower() for token in
                            ("429", "500", "502", "503", "504", "deadline", "temporarily"))
            if attempt == 2 or not retryable:
                raise
            sleep(2 ** (attempt + 1))
    return check_weather(pixels_to_weather(pixels, block), expected_keys)


def read_weather_for_panel(path, weather_keys, days):
    frame = pd.read_parquet(path, columns=GFS_COLUMNS)
    frame["issue_utc"] = pd.to_datetime(frame["issue_utc"], utc=True)
    frame = frame.loc[frame["issue_utc"].between(days[0], days[-1])]
    checked = check_weather(frame, weather_keys)
    print(f"Loaded prepared GFS: {path}")
    return checked


def load_weather(weather_keys):
    days = pd.DatetimeIndex(weather_keys["issue_utc"].unique()).sort_values()
    if weather_path.is_file():
        return read_weather_for_panel(weather_path, weather_keys, days)

    blocks = [days[i:i + BLOCK_DAYS] for i in range(0, len(days), BLOCK_DAYS)]
    ee = None
    parts = []
    for i, block in enumerate(blocks, 1):
        path = cache_dir / f"{block[0]:%Y%m%d}_{block[-1]:%Y%m%d}.parquet"
        expected = weather_keys.loc[weather_keys["issue_utc"].between(block[0], block[-1])]
        try:
            if path.is_file():
                part = check_weather(pd.read_parquet(path), expected)
            else:
                legacy = read_legacy_gfs_block(legacy_cache_dir / path.name, (GFS_FEATURES,))
                if legacy is not None:
                    part = check_weather(legacy, expected)
                else:
                    if ee is None:
                        ee = open_earth_engine()
                    part = fetch_block(ee, block, expected)
                atomic_parquet(part, path)
        except Exception as exc:
            print(f"GFS day 1 block {block[0]:%Y-%m-%d} to {block[-1]:%Y-%m-%d}: "
                  f"{type(exc).__name__}: {exc}", flush=True)
            raise
        if part[GFS_FEATURES].isna().all(axis=None):
            print(f"No GFS day 1 values for {block[0]:%Y-%m-%d} to {block[-1]:%Y-%m-%d}.", flush=True)
        parts.append(part)
        if i == 1 or i % 10 == 0 or i == len(blocks):
            print(f"GFS: {i}/{len(blocks)} blocks through {block[-1]:%Y-%m-%d}", flush=True)
    weather = check_weather(pd.concat(parts, ignore_index=True), weather_keys)
    empty = [name for name in GFS_FEATURES if weather[name].isna().all()]
    if empty:
        raise ValueError(f"No usable values for these day 1 GFS features: {', '.join(empty)}.")
    atomic_parquet(weather, weather_path)
    return weather

day1_keys = grid_table.loc[grid_table["issue_utc"].ge(GFS_DAY1_FIRST_ISSUE), KEYS]
weather_day1 = load_weather(day1_keys)
print(f"Saved {len(weather_day1):,} rows to {weather_path.name}.")

In [ ]:
# 3. Download GFS features for days 2 and 3.
KEYS_3D = KEYS
EXTRA_WEATHER_3D = {
    day: [f"gfs_day{day}_{name}" for name in GFS_FEATURES] for day in (2, 3)
}
WEATHER_3D_COLUMNS = [*KEYS, "gfs_run_utc", *EXTRA_WEATHER_3D[2], *EXTRA_WEATHER_3D[3]]

def check_gfs_3d(weather, expected):
    weather = weather[WEATHER_3D_COLUMNS].copy()
    for name in ("issue_utc", "gfs_run_utc"):
        weather[name] = pd.to_datetime(weather[name], utc=True).astype("datetime64[ns, UTC]")
    if len(weather) != len(expected) or weather.duplicated(KEYS_3D).any():
        raise ValueError("The 3-day GFS data must provide exactly one row per requested issue and cell.")
    weather = expected.merge(weather, on=KEYS_3D, how="left", validate="one_to_one")
    if not weather["gfs_run_utc"].eq(weather["issue_utc"] - pd.Timedelta(hours=6)).all():
        raise ValueError("Each horizon must use the GFS run from 18 UTC on the previous day.")
    for names in EXTRA_WEATHER_3D.values():
        clean_gfs_values(weather, names)
    return weather


def fetch_gfs_3d_block(ee, dates, ys_3d, xs_3d):
    cell_deg = 0.25
    affine = [cell_deg, 0, float(xs_3d[0]) * cell_deg, 0, -cell_deg, float(ys_3d[-1] + 1) * cell_deg]
    grid = {
        "dimensions": {"width": len(xs_3d), "height": len(ys_3d)},
        "affineTransform": dict(zip(
            ["scaleX", "shearX", "translateX", "shearY", "scaleY", "translateY"], affine,
        )),
        "crsCode": "EPSG:4326",
    }
    archive = ee.ImageCollection(GFS_DATASET)
    state_bands = [
        "temperature_2m_above_ground", "relative_humidity_2m_above_ground",
        "u_component_of_wind_10m_above_ground", "v_component_of_wind_10m_above_ground",
    ]
    rain_band = "total_precipitation_surface"
    missing = -9999.0

    def issue_image(issue):
        run = issue - pd.Timedelta(hours=6)
        cycle = archive.filter(ee.Filter.eq("creation_time", run.value // 10**6))

        def step(lead, bands, names):
            valid = run + pd.Timedelta(hours=lead)
            matched = cycle.filter(ee.Filter.eq("forecast_hours", lead)).filter(
                ee.Filter.eq("forecast_time", valid.value // 10**6)
            )
            empty = ee.Image.constant([0] * len(names)).rename(names).selfMask().toFloat()
            image = ee.Image(matched.first()).select(bands, names).toFloat()
            image = image.resample("bilinear").reproject(crs="EPSG:4326", crsTransform=affine)
            return ee.Image(ee.Algorithms.If(matched.size().eq(1), image, empty))

        def state(lead):
            image = step(lead, state_bands, ["temp", "rh", "u", "v"])
            wind = image.select("u").hypot(image.select("v")).rename("wind")
            return image.select(["temp", "rh"]).addBands(wind)

        def aggregate(collection, band, reducer, count):
            selected = collection.select(band)
            return getattr(selected, reducer)().updateMask(selected.count().eq(count))

        results = []
        for day, names in EXTRA_WEATHER_3D.items():
            start = 6 + 24 * (day - 1)
            states = ee.ImageCollection.fromImages([state(lead) for lead in range(start, start + 24, 3)])
            # At multiples of six, the GFS rain band is a six-hour accumulation.
            rain = ee.ImageCollection.fromImages([
                step(lead, [rain_band], ["rain"]) for lead in range(start + 6, start + 25, 6)
            ])
            reductions = [
                aggregate(states, "temp", "max", 8), aggregate(states, "rh", "min", 8),
                aggregate(states, "wind", "max", 8), aggregate(rain, "rain", "sum", 4),
            ]
            results.extend(band.rename(f"d{issue:%Y%m%d}_{name}") for band, name in zip(reductions, names))
        return ee.Image.cat(*results).toFloat().unmask(missing)

    request = {
        "expression": ee.Image.cat(*[issue_image(date) for date in dates]),
        "fileFormat": "NUMPY_NDARRAY",
        "grid": grid,
    }
    for attempt in range(3):
        try:
            pixels = ee.data.computePixels(request)
            break
        except ee.EEException as error:
            retryable = any(s in str(error).lower() for s in (
                "429", "500", "502", "503", "504", "deadline", "temporarily", "timed out",
            ))
            if attempt == 2 or not retryable:
                raise
            sleep(2 ** (attempt + 1))

    if pixels.shape != (len(ys_3d), len(xs_3d)):
        raise ValueError(f"Unexpected GFS pixel grid: {pixels.shape}")
    cells = pd.MultiIndex.from_product([ys_3d[::-1], xs_3d], names=KEYS_3D[1:]).to_frame(index=False)
    parts = []
    for issue in dates:
        frame = cells.assign(issue_utc=issue, gfs_run_utc=issue - pd.Timedelta(hours=6))
        for name in WEATHER_3D_COLUMNS[4:]:
            values = pixels[f"d{issue:%Y%m%d}_{name}"].astype(np.float32).ravel()
            frame[name] = np.where(values == missing, np.nan, values)
        parts.append(frame)
    return pd.concat(parts, ignore_index=True)


def load_gfs_3d(data, dates, ys_3d, xs_3d):
    final_path = GFS_QC_ROOT / "gfs_days_2_3.parquet"
    expected = data[KEYS_3D]
    if "gfs_run_utc" in data:
        day1_runs = pd.to_datetime(data["gfs_run_utc"], utc=True)
        wrong = day1_runs.notna() & day1_runs.ne(data["issue_utc"] - pd.Timedelta(hours=6))
        if wrong.any():
            raise ValueError("Day 1 in the prepared panel uses a different GFS run.")

    if final_path.is_file():
        weather = check_gfs_3d(pd.read_parquet(final_path), expected)
    else:
        if not (
            np.array_equal(ys_3d, np.arange(152, 168))
            and np.array_equal(xs_3d, np.arange(-24, -6))
        ):
            raise ValueError("The GFS downloader is configured for the central Spain 0.25° grid.")
        ee = None
        parts = []
        batch_size = 7
        batches = (len(dates) + batch_size - 1) // batch_size
        for first in range(0, len(dates), batch_size):
            block = dates[first:first + batch_size]
            path = GFS_QC_ROOT / "gfs_cache_v1" / f"{block[0]:%Y%m%d}_{block[-1]:%Y%m%d}.parquet"
            block_keys = expected.loc[expected["issue_utc"].between(block[0], block[-1])]
            if path.is_file():
                part = check_gfs_3d(pd.read_parquet(path), block_keys)
            else:
                legacy = read_legacy_gfs_block(
                    OUTPUT_ROOT / "gfs_cache_v1" / path.name, EXTRA_WEATHER_3D.values(),
                )
                if legacy is not None:
                    part = check_gfs_3d(legacy, block_keys)
                else:
                    if ee is None:
                        ee = open_earth_engine()
                    part = check_gfs_3d(fetch_gfs_3d_block(ee, block, ys_3d, xs_3d), block_keys)
                atomic_parquet(part, path)
            if part[WEATHER_3D_COLUMNS[4:]].isna().all(axis=None):
                print(f"No GFS days 2–3 values for {block[0]:%Y-%m-%d} to {block[-1]:%Y-%m-%d}.", flush=True)
            parts.append(part)
            if len(parts) == 1 or len(parts) % 10 == 0 or len(parts) == batches:
                print(f"GFS days 2–3: {len(parts)}/{batches} blocks", flush=True)
        weather = check_gfs_3d(pd.concat(parts, ignore_index=True), expected)
    if weather[WEATHER_3D_COLUMNS[4:]].isna().all().any():
        raise ValueError("At least one additional GFS weather feature has no usable values.")
    if not final_path.is_file():
        atomic_parquet(weather, final_path)
    return weather

day23_issues = issue_days[issue_days >= GFS_DAY23_FIRST_ISSUE]
day23_keys = grid_table.loc[grid_table["issue_utc"].ge(GFS_DAY23_FIRST_ISSUE), KEYS]
weather_days_2_3 = load_gfs_3d(day23_keys, day23_issues, ys, xs)
print(f"Saved {len(weather_days_2_3):,} rows to gfs_days_2_3.parquet.")

## Terrain, land cover, and moisture

NASADEM provides fixed terrain features for each cell. CORINE 2012 and 2018 are exported
as separate land cover snapshots. The combined table uses the original CORINE 2018 layer
for all issue dates.

SMAP uses the most recent eligible complete UTC day and a trailing seven-day mean.
A source day must have ended at least seven days before the forecast issue. NDMI uses
quality-screened eight-day MODIS composites with the original publication delay.

In [ ]:
# 4. Download terrain and land-cover features.
def cell_regions(ee, cells):
    return ee.FeatureCollection([
        ee.Feature(
            ee.Geometry.Rectangle(
                [x * CELL_DEG, y * CELL_DEG, (x + 1) * CELL_DEG, (y + 1) * CELL_DEG],
                proj="EPSG:4326",
                geodesic=False,
            ),
            {"cell_y": int(y), "cell_x": int(x)},
        )
        for y, x in cells[CELL_KEYS].itertuples(index=False, name=None)
    ])


def download_terrain(cells):
    ee = open_earth_engine()
    dem = ee.Image(DEM_DATASET).select("elevation")
    terrain_image = dem.addBands(ee.Terrain.slope(dem)).rename(TERRAIN_FEATURES)
    stats = terrain_image.reduceRegions(collection=cell_regions(ee, cells),
                                        reducer=ee.Reducer.mean(), scale=90)
    return ee.data.computeFeatures({"expression": stats, "fileFormat": "PANDAS_DATAFRAME"})


def download_land_cover(cells, year):
    ee = open_earth_engine()
    clc = ee.Image(f"{CORINE_COLLECTION}/{year}").select("landcover")
    cover_bands = ee.Image.cat(*[
        clc.remap(classes, [1] * len(classes), defaultValue=0).rename(name)
        for name, classes in COVER_GROUPS.items()
    ])
    stats = cover_bands.reduceRegions(collection=cell_regions(ee, cells), reducer=ee.Reducer.mean(),
                                      crs="EPSG:3035", scale=100)
    return ee.data.computeFeatures({"expression": stats, "fileFormat": "PANDAS_DATAFRAME"})


def check_table(frame, expected, keys, values):
    frame = frame[[*keys, *values]].copy()
    if "issue_utc" in keys:
        frame["issue_utc"] = pd.to_datetime(frame["issue_utc"], utc=True).astype("datetime64[ns, UTC]")
    reference = pd.MultiIndex.from_frame(expected[keys])
    found = pd.MultiIndex.from_frame(frame[keys])
    if len(found) != len(reference) or not found.is_unique or not found.sort_values().equals(reference.sort_values()):
        raise ValueError(f"Feature table keys do not match the expected grid for {values}.")
    return frame

def read_or_build_feature(path, expected, keys, values, build):
    if path.is_file():
        print(f"Using {path.name} from Drive.", flush=True)
        table = pd.read_parquet(path)
    else:
        table = build()
    return check_table(table, expected, keys, values)

def load_static_feature(path, values, build):
    if path.is_file():
        return check_table(pd.read_parquet(path), grid_cells, CELL_KEYS, values)
    table = check_table(build(), grid_cells, CELL_KEYS, values)
    atomic_parquet(table, path)
    return table


terrain = load_static_feature(
    OUTPUT_ROOT / "elevation_by_cell.parquet", TERRAIN_FEATURES, lambda: download_terrain(grid_cells),
)
land_cover_snapshots = {}
for year in CLC_YEARS:
    land_cover_snapshots[year] = load_static_feature(
        OUTPUT_ROOT / f"corine_{year}_by_cell.parquet", COVER_FEATURES,
        lambda: download_land_cover(grid_cells, year),
    )
land_cover = land_cover_snapshots[CLC_YEAR]
print(f"Terrain and land-cover tables are ready for {len(grid_cells)} cells.")

In [ ]:
# 5. Download SMAP and MODIS inputs and compute moisture features.
def fetch_smap_block(ee, smap, projection, regions, cells, days):
    # Keep the band present on archive days with no usable frames.
    masked = ee.Image.constant(0).rename("sm_surface").toFloat().selfMask()
    fallback = ee.ImageCollection.fromImages([masked])

    images = []
    band_dates = {}
    for day in days:
        start = day.strftime("%Y-%m-%d")
        end = (day + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        day_id = day.strftime("d%Y%m%d")
        frames = smap.filterDate(start, end).merge(fallback)
        daily = frames.mean().updateMask(frames.count().eq(8))
        images.append(daily.toFloat().set("system:index", day_id))
        band_dates[f"{day_id}_sm_surface"] = day

    stats = ee.ImageCollection.fromImages(images).toBands().reduceRegions(
        collection=regions,
        reducer=ee.Reducer.mean(),
        crs=projection["crs"],
        crsTransform=projection["transform"],
    )
    wide = ee.data.computeFeatures({"expression": stats, "fileFormat": "PANDAS_DATAFRAME"})
    expected_cells = pd.MultiIndex.from_frame(cells)
    returned_cells = pd.MultiIndex.from_frame(wide[CELL_KEYS])
    if not returned_cells.is_unique or not expected_cells.sort_values().equals(returned_cells.sort_values()):
        raise ValueError("SMAP grid keys differ from the forecast grid.")

    daily = wide.reindex(columns=[*CELL_KEYS, *band_dates]).melt(
        id_vars=CELL_KEYS, var_name="band", value_name="sm_surface",
    )
    daily[SMAP_SOURCE_DAY] = daily.pop("band").map(band_dates)
    daily["sm_surface"] = pd.to_numeric(daily["sm_surface"], errors="raise")
    return daily


def load_smap_daily(cells, days):
    cell_pairs = tuple(cells.itertuples(index=False, name=None))
    signature = repr((SMAP_DATASET, CELL_DEG, cell_pairs, "daily_complete_v1"))
    cache_dir = OUTPUT_ROOT / "smap_daily_cache" / sha256(signature.encode()).hexdigest()[:12]

    ee = None
    parts = []
    for i in range(0, len(days), SMAP_BLOCK_DAYS):
        block = days[i:i + SMAP_BLOCK_DAYS]
        path = cache_dir / f"{block[0]:%Y%m%d}_{block[-1]:%Y%m%d}.parquet"
        if path.is_file():
            part = pd.read_parquet(path)
        else:
            if ee is None:
                ee = open_earth_engine()
                smap = ee.ImageCollection(SMAP_DATASET).select("sm_surface")
                projection = ee.Image(smap.first()).projection().getInfo()
                regions = cell_regions(ee, cells)
            try:
                part = fetch_smap_block(ee, smap, projection, regions, cells, block)
            except ee.EEException as error:
                raise RuntimeError(f"SMAP {block[0]:%Y-%m-%d} – {block[-1]:%Y-%m-%d}: {error}") from error
            atomic_parquet(part, path)
        parts.append(part)
        if i == 0 or len(parts) % 10 == 0 or i + len(block) == len(days):
            print(f"SMAP: {i + len(block)}/{len(days)} days", flush=True)
    return pd.concat(parts, ignore_index=True)


def add_smap_features(panel):
    if panel.empty or panel["issue_utc"].isna().any() or panel.duplicated(KEYS).any():
        raise ValueError("Forecast dates and cell keys must be non-empty and unique.")
    cells = panel[CELL_KEYS].drop_duplicates().sort_values(CELL_KEYS).reset_index(drop=True)
    issue_days = pd.DatetimeIndex(panel["issue_utc"].unique()).sort_values()
    if not issue_days.equals(issue_days.normalize()) or str(issue_days.tz) != "UTC":
        raise ValueError("Expected daily issues at 00:00 UTC.")

    # Allow the full publication lag after the final hour of each source day.
    source_days = issue_days - pd.Timedelta(days=SMAP_PUBLICATION_LAG_DAYS + 1)
    daily_range = pd.date_range(
        source_days[0] - pd.Timedelta(days=SMAP_HISTORY_DAYS - 1), source_days[-1], freq="D",
    )
    available_days = daily_range[daily_range >= SMAP_ARCHIVE_START]
    daily = load_smap_daily(cells, available_days)
    daily[SMAP_SOURCE_DAY] = pd.to_datetime(daily[SMAP_SOURCE_DAY], utc=True)
    grid = daily.pivot(index=SMAP_SOURCE_DAY, columns=CELL_KEYS, values="sm_surface").reindex(
        index=daily_range, columns=pd.MultiIndex.from_frame(cells),
    )
    if (grid.lt(0) | grid.gt(1)).to_numpy().any():
        raise ValueError("SMAP moisture must be a fraction in [0, 1].")
    weekly = grid.rolling(SMAP_HISTORY_DAYS, min_periods=SMAP_HISTORY_DAYS).mean()

    table = pd.DataFrame({
        "issue_utc": issue_days.repeat(len(cells)),
        "cell_y": np.tile(cells["cell_y"].to_numpy(), len(issue_days)),
        "cell_x": np.tile(cells["cell_x"].to_numpy(), len(issue_days)),
        SMAP_SOURCE_DAY: source_days.repeat(len(cells)),
        SMAP_FEATURES[0]: grid.loc[source_days].to_numpy(dtype=np.float32).ravel(),
        SMAP_FEATURES[1]: weekly.loc[source_days].to_numpy(dtype=np.float32).ravel(),
    })
    table[SMAP_SOURCE_DAY] = table[SMAP_SOURCE_DAY].where(table[SMAP_FEATURES[0]].notna())
    joined = panel.drop(columns=[SMAP_SOURCE_DAY, *SMAP_FEATURES], errors="ignore").join(
        table.set_index(KEYS), on=KEYS, how="left", validate="one_to_one",
    )
    coverage = joined[SMAP_FEATURES].notna().groupby(joined["year"]).mean()
    if joined[SMAP_FEATURES].isna().all().any():
        raise ValueError("At least one SMAP feature has no usable values in the requested period.")
    atomic_parquet(
        joined[[*KEYS, SMAP_SOURCE_DAY, *SMAP_FEATURES]],
        OUTPUT_ROOT / f"smap_v008_lag{SMAP_PUBLICATION_LAG_DAYS}_features.parquet",
    )
    return joined, coverage


def ndmi_image(image):
    state = image.select("StateQA")
    quality = image.select("QA")
    nir = image.select("sur_refl_b02")
    swir = image.select("sur_refl_b06")

    cloud_and_snow_flags = (1 << 2) | (3 << 8) | (1 << 10) | (1 << 12) | (1 << 13) | (1 << 15)
    band_quality_flags = (15 << 6) | (15 << 22)
    valid = (
        state.bitwiseAnd(3).eq(0)
        .And(state.rightShift(3).bitwiseAnd(7).eq(1))
        .And(state.bitwiseAnd(cloud_and_snow_flags).eq(0))
        .And(quality.bitwiseAnd(3).lt(2))
        .And(quality.bitwiseAnd(band_quality_flags).eq(0))
        .And(nir.gte(0)).And(nir.lte(16000))
        .And(swir.gte(0)).And(swir.lte(16000))
        .And(nir.add(swir).gt(0))
    )
    ndmi = (
        image.normalizedDifference(["sur_refl_b02", "sur_refl_b06"])
        .updateMask(valid)
        .rename("ndmi_mean")
    )
    coverage = ndmi.mask().unmask(0, sameFootprint=False).rename(NDMI_COVERAGE)
    return ndmi.addBands(coverage).toFloat()


def fetch_ndmi_block(ee, modis, projection, regions, cells, block):
    images = []
    bands_by_day = {}
    # Name the output bands directly, independently of MODIS asset IDs.
    for day in block:
        start = day.strftime("%Y-%m-%d")
        stop = (day + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        image = ee.Image(modis.filterDate(start, stop).first())
        prefix = day.strftime("d%Y%m%d_")
        names = [prefix + "ndmi_mean", prefix + NDMI_COVERAGE]
        bands_by_day[day] = names
        images.append(ndmi_image(image).rename(names))

    stats = ee.Image.cat(*images).reduceRegions(
        collection=regions, reducer=ee.Reducer.mean(),
        crs=projection["crs"], crsTransform=projection["transform"],
    )
    wide = ee.data.computeFeatures({"expression": stats, "fileFormat": "PANDAS_DATAFRAME"})
    expected = pd.MultiIndex.from_frame(cells).sort_values()
    received = pd.MultiIndex.from_frame(wide[CELL_KEYS]).sort_values()
    if not received.equals(expected):
        raise ValueError("Earth Engine returned different NDMI cell keys.")

    parts = []
    for day, (mean_column, coverage_column) in bands_by_day.items():
        if coverage_column not in wide.columns:
            raise ValueError(
                f"Earth Engine did not return {coverage_column}. "
                f"Returned columns: {wide.columns.tolist()}"
            )
        coverage = pd.to_numeric(wide[coverage_column])
        mean = pd.to_numeric(wide.get(mean_column, pd.Series(float("nan"), index=wide.index)))
        if not coverage.between(0, 1).all():
            raise ValueError(f"Invalid NDMI coverage in {coverage_column}.")
        if (coverage.gt(0) & mean.isna()).any():
            raise ValueError(f"NDMI coverage is positive without a value for {mean_column}.")
        parts.append(wide[CELL_KEYS].assign(**{
            NDMI_SOURCE_DATE: day,
            "ndmi_mean": mean,
            NDMI_COVERAGE: coverage,
        }))
    return pd.concat(parts, ignore_index=True)


def load_ndmi_composites(cells, earliest, latest):
    ee = open_earth_engine()
    modis = ee.ImageCollection(NDMI_DATASET).filterDate(
        earliest.strftime("%Y-%m-%d"), (latest + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
    ).select(["sur_refl_b02", "sur_refl_b06", "StateQA", "QA"])
    dates = pd.DatetimeIndex(pd.to_datetime(
        modis.aggregate_array("system:time_start").getInfo(), unit="ms", utc=True,
    )).sort_values()
    if dates.empty or not dates.is_unique or not dates.equals(dates.normalize()):
        raise ValueError("Expected unique MODIS composite dates at midnight.")

    cell_pairs = list(cells.itertuples(index=False, name=None))
    signature = repr((NDMI_DATASET, CELL_DEG, cell_pairs, "ndmi_qa_v1"))
    cache_dir = OUTPUT_ROOT / "ndmi_cache" / sha256(signature.encode()).hexdigest()[:12]
    projection = None
    parts = []
    for i in range(0, len(dates), NDMI_BLOCK_SIZE):
        block = dates[i:i + NDMI_BLOCK_SIZE]
        block_id = sha256(repr(tuple(block.asi8)).encode()).hexdigest()[:10]
        path = cache_dir / f"{block[0]:%Y%m%d}_{block[-1]:%Y%m%d}_{block_id}.parquet"
        if path.is_file():
            part = pd.read_parquet(path)
        else:
            if projection is None:
                projection = ee.Image(modis.first()).select("sur_refl_b02").projection().getInfo()
                regions = cell_regions(ee, cells)
            try:
                part = fetch_ndmi_block(ee, modis, projection, regions, cells, block)
            except ee.EEException as error:
                raise RuntimeError(f"NDMI {block[0]:%Y-%m-%d} – {block[-1]:%Y-%m-%d}: {error}") from error
            atomic_parquet(part, path)
        parts.append(part)
        if i == 0 or len(parts) % 10 == 0 or i + len(block) == len(dates):
            print(f"NDMI: {i + len(block)}/{len(dates)} composites", flush=True)
    return pd.concat(parts, ignore_index=True)


def add_ndmi_features(panel):
    keys = panel[KEYS]
    if keys.empty or keys.isna().any().any() or keys.duplicated().any():
        raise ValueError("Forecast dates and cell keys must be non-empty and unique.")
    issue_dates = pd.DatetimeIndex(keys["issue_utc"])
    if str(issue_dates.tz) != "UTC" or not issue_dates.equals(issue_dates.normalize()):
        raise ValueError("Expected daily issues at 00:00 UTC.")

    delay = pd.Timedelta(days=NDMI_COMPOSITE_DAYS + NDMI_PUBLICATION_BUFFER_DAYS)
    fallback = pd.Timedelta(days=NDMI_MAX_FALLBACK_DAYS)
    cells = keys[CELL_KEYS].drop_duplicates().sort_values(CELL_KEYS).reset_index(drop=True)
    composites = load_ndmi_composites(cells, issue_dates.min() - delay - fallback, issue_dates.max() - delay)
    composites = composites.astype(cells.dtypes.to_dict())
    source_dates = pd.to_datetime(composites[NDMI_SOURCE_DATE], utc=True).astype("datetime64[ns, UTC]")
    composites = composites.assign(**{NDMI_SOURCE_DATE: source_dates, NDMI_AVAILABLE_DATE: source_dates + delay})
    if composites.duplicated([*CELL_KEYS, NDMI_SOURCE_DATE]).any():
        raise ValueError("NDMI contains duplicate cell-composite keys.")

    usable = composites.loc[
        composites[NDMI_COVERAGE].between(NDMI_MIN_VALID_FRACTION, 1)
        & composites["ndmi_mean"].between(-1, 1)
    ]
    matched = pd.merge_asof(
        keys.sort_values("issue_utc"), usable.sort_values(NDMI_AVAILABLE_DATE),
        left_on="issue_utc", right_on=NDMI_AVAILABLE_DATE, by=CELL_KEYS,
        direction="backward", tolerance=fallback,
    )
    columns = [*VEGETATION_FEATURES, NDMI_SOURCE_DATE, NDMI_AVAILABLE_DATE, NDMI_COVERAGE]
    joined = panel.drop(columns=columns, errors="ignore").join(
        matched.set_index(KEYS)[columns], on=KEYS, how="left", validate="one_to_one",
    )
    coverage = joined["ndmi_mean"].notna().groupby(joined["year"]).mean().rename("ndmi_coverage")
    if not joined["ndmi_mean"].notna().any():
        raise ValueError("There are no valid NDMI values for these cells and dates.")
    atomic_parquet(joined[[*KEYS, *columns]], OUTPUT_ROOT / "ndmi_features.parquet")
    return joined, coverage

smap_columns = [SMAP_SOURCE_DAY, *SMAP_FEATURES]
smap_features = read_or_build_feature(
    OUTPUT_ROOT / f"smap_v008_lag{SMAP_PUBLICATION_LAG_DAYS}_features.parquet",
    grid_table, KEYS, smap_columns, lambda: add_smap_features(grid_table)[0],
)
ndmi_columns = [*VEGETATION_FEATURES, NDMI_SOURCE_DATE, NDMI_AVAILABLE_DATE, NDMI_COVERAGE]
ndmi_features = read_or_build_feature(
    OUTPUT_ROOT / "ndmi_features.parquet",
    grid_table, KEYS, ndmi_columns, lambda: add_ndmi_features(grid_table)[0],
)

## Fire Weather Index

The FWI uses GFS temperature, relative humidity, wind, and precipitation at noon UTC.
The calculation uses a 120-day initialization period. It begins with the earliest eligible
GFS weather when the requested dates precede the archive. The first 120 days of archive
weather are used only to initialize the moisture codes. The FWI equations use humidity
capped at 100%. This cap applies to the calculation only; the cached source inputs and
forecast weather tables keep their original readings above 100%.

In [ ]:
# 6. Download GFS inputs for FWI and calculate the index.
def fwi_day_image(ee, archive, issue):
    run_18 = issue - pd.Timedelta(hours=RUN_AGE_HOURS)
    noon = issue + pd.Timedelta(hours=12)

    def select_at(valid, latest_run, source_bands, names):
        candidates = []
        for backup_hours in (24, 6, 0):
            run = latest_run - pd.Timedelta(hours=backup_hours)
            lead = int((valid - run) / pd.Timedelta(hours=1))
            matching = (archive.filter(ee.Filter.eq("creation_time", run.value // 10**6))
                        .filter(ee.Filter.eq("forecast_time", valid.value // 10**6))
                        .filter(ee.Filter.eq("forecast_hours", lead)))
            empty = ee.Image.constant([0] * len(names)).rename(names).selfMask().toFloat()
            forecast = (ee.Image(matching.first()).select(source_bands, names).toFloat()
                        .resample("bilinear").reproject(crs="EPSG:4326", crsTransform=transform))
            forecast = forecast.updateMask(forecast.mask().reduce(ee.Reducer.min()))
            candidates.append(ee.Image(ee.Algorithms.If(matching.size().eq(1), forecast, empty)))
        return ee.ImageCollection.fromImages(candidates).mosaic()

    state = select_at(
        noon, run_18, [*FWI_GFS_BANDS, FWI_GFS_RAIN], ["temperature", "humidity", "u", "v", "rain"],
    )
    wind = state.select("u").hypot(state.select("v")).multiply(3.6)
    rain_bands = [FWI_GFS_RAIN]
    rain_intervals = [
        select_at(run_18, run_18 - pd.Timedelta(hours=6), rain_bands, ["rain"]),
        *[select_at(run_18 + pd.Timedelta(hours=lead), run_18, rain_bands, ["rain"])
          for lead in (6, 12)],
        state.select("rain"),
    ]
    # Adding the bands directly keeps the daily total masked if any interval is missing.
    rain = rain_intervals[0].add(rain_intervals[1]).add(rain_intervals[2]).add(rain_intervals[3])
    bands = state.select(["temperature", "humidity"]).addBands([wind, rain])
    return bands.toFloat().unmask(MISSING_PIXEL).rename([
        f"d{issue:%Y%m%d}_{name}" for name in FWI_WEATHER
    ])


def fetch_fwi_weather(ee, block):
    archive = ee.ImageCollection(GFS_DATASET).filter(ee.Filter.inList(
        "forecast_hours", list(range(6, 43, 6)),
    ))
    stack = ee.Image.cat(*[fwi_day_image(ee, archive, day) for day in block])
    request = {"expression": stack, "fileFormat": "NUMPY_NDARRAY", "grid": pixel_grid}
    for attempt in range(3):
        try:
            pixels = ee.data.computePixels(request)
            break
        except ee.EEException as error:
            retryable = any(token in str(error).lower() for token in (
                "429", "500", "502", "503", "504", "deadline", "temporarily", "timed out",
            ))
            if attempt == 2 or not retryable:
                raise
            sleep(2 ** (attempt + 1))

    if pixels.shape != (len(ys), len(xs)):
        raise ValueError(f"Unexpected FWI grid shape: {pixels.shape}")
    grid_cells = pd.MultiIndex.from_product([ys[::-1], xs], names=KEYS[1:]).to_frame(index=False)
    frames = []
    for day in block:
        frame = grid_cells.assign(issue_utc=day)
        for name in FWI_WEATHER:
            values = pixels[f"d{day:%Y%m%d}_{name}"].astype(np.float32).ravel()
            frame[name] = np.where(values == MISSING_PIXEL, np.nan, values)
        frames.append(frame[[*KEYS, *FWI_WEATHER]])
    return pd.concat(frames, ignore_index=True)


def load_fwi_weather(days):
    signature = repr((GFS_DATASET, tuple(xs), tuple(ys), transform, RUN_AGE_HOURS, "noon_utc_v1"))
    cache_id = sha256(signature.encode()).hexdigest()[:12]
    cache_dir = OUTPUT_ROOT / "fwi_gfs_cache" / cache_id
    ee = None
    blocks = []
    for i in range(0, len(days), FWI_BLOCK_DAYS):
        block = days[i:i + FWI_BLOCK_DAYS]
        path = cache_dir / f"{block[0]:%Y%m%d}_{block[-1]:%Y%m%d}.parquet"
        if path.is_file():
            frame = pd.read_parquet(path)
        else:
            if ee is None:
                ee = open_earth_engine()
            frame = fetch_fwi_weather(ee, block)
            atomic_parquet(frame, path)
        blocks.append(frame)
        finished = i + len(block)
        if i == 0 or len(blocks) % 10 == 0 or finished == len(days):
            print(f"GFS for FWI: {finished}/{len(days)} days", flush=True)
    return pd.concat(blocks, ignore_index=True)


def calculate_fwi(weather, days):
    import xarray as xr
    from xclim.indices.fire import cffwis_indices

    weather = weather.assign(issue_utc=pd.to_datetime(weather["issue_utc"], utc=True))
    weather = clean_gfs_values(weather, FWI_WEATHER).set_index(KEYS, verify_integrity=True)
    grid = pd.MultiIndex.from_product([days, ys, xs], names=KEYS)
    if len(grid.difference(weather.index)) or len(weather.index.difference(grid)):
        raise ValueError("FWI weather does not cover the expected dates and cells.")

    values = weather.reindex(grid)[FWI_WEATHER].to_numpy(dtype=float, copy=True).reshape(len(days), -1, 4)
    # The FWI equations include fractional powers of (100 - RH).
    values[:, :, 1] = np.minimum(values[:, :, 1], 100)
    valid = np.isfinite(values).all(axis=2)
    fwi = np.full(valid.shape, np.nan, dtype=np.float32)
    latitudes = (np.repeat(ys, len(xs)) + 0.5) * CELL_DEG
    dates = days.tz_localize(None)

    # Group cells with the same missing days so each run can use contiguous valid weather.
    _, groups = np.unique(valid.T, axis=0, return_inverse=True)
    for group in np.unique(groups):
        cells = np.flatnonzero(groups == group)
        usable_days = np.flatnonzero(valid[:, cells[0]])
        segments = np.split(usable_days, np.flatnonzero(np.diff(usable_days) > 1) + 1)
        previous_codes = {}
        lat = xr.DataArray(latitudes[cells], dims="cell", attrs={"units": "degrees_north"})
        for segment in segments:
            if not len(segment):
                continue
            data = {
                arg: xr.DataArray(values[segment[:, None], cells, idx], dims=("time", "cell"),
                                  coords={"time": dates[segment]}, attrs={"units": unit})
                for idx, (arg, unit) in enumerate((
                    ("tas", "degC"), ("hurs", "%"), ("sfcWind", "km/h"), ("pr", "mm/day"),
                ))
            }
            indices = cffwis_indices(**data, lat=lat, season_method=None, **previous_codes)
            fwi[np.ix_(segment, cells)] = indices.FWI.transpose("time", "cell").values
            # Missing days retain the last modeled moisture state and have no FWI value.
            previous_codes = {name.lower() + "0": getattr(indices, name).isel(time=-1, drop=True).copy()
                              for name in ("FFMC", "DMC", "DC")}

    return pd.DataFrame({"fwi": fwi.ravel()}, index=grid).reset_index()


def add_fwi(panel):
    panel_keys = panel[KEYS]
    if panel_keys.empty or panel_keys.isna().any().any() or panel_keys.duplicated().any():
        raise ValueError("Forecast dates and cell keys must be non-empty and unique.")
    if not (panel_keys["cell_y"].isin(ys) & panel_keys["cell_x"].isin(xs)).all():
        raise ValueError("Forecast cells do not match the GFS grid.")
    issues = pd.DatetimeIndex(panel["issue_utc"].unique()).sort_values()
    if str(issues.tz) != "UTC" or not issues.equals(issues.normalize()):
        raise ValueError("Expected daily issues at 00:00 UTC.")
    first_weather_day = max(GFS_DAY1_FIRST_ISSUE, issues[0] - pd.Timedelta(days=FWI_WARMUP_DAYS))
    days = pd.date_range(first_weather_day, issues[-1], freq="D")
    if find_spec("xclim") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "xclim==0.59.1"])
    weather = load_fwi_weather(days)
    fwi_by_day = calculate_fwi(weather, days)
    fwi_by_day = fwi_by_day.loc[
        fwi_by_day["issue_utc"].isin(issues) & fwi_by_day["issue_utc"].ge(FWI_FIRST_ISSUE)
    ]
    joined = panel.drop(columns="fwi", errors="ignore").join(
        fwi_by_day.set_index(KEYS), on=KEYS, how="left", validate="one_to_one",
    )
    coverage = joined["fwi"].notna().groupby(joined["year"]).mean().rename("fwi_coverage")
    if joined["fwi"].isna().all():
        raise ValueError("FWI has no usable values after the 120-day initialization period.")
    atomic_parquet(joined[[*KEYS, "fwi"]], GFS_QC_ROOT / "fwi_features.parquet")
    return joined, coverage

fwi_features = read_or_build_feature(
    GFS_QC_ROOT / "fwi_features.parquet", grid_table, KEYS, ["fwi"], lambda: add_fwi(grid_table)[0],
)

## Export the environmental feature table

Join the weather, terrain, land cover, moisture, and FWI by forecast issue and grid cell.
The coverage report shows the fraction of rows with a usable value for each feature and
year. The table can be joined later with fire detections and labels using the three grid keys.

In [ ]:
# 7. Save the complete environmental grid and the annual coverage report.
ENVIRONMENT_FEATURES = [
    *TERRAIN_FEATURES, *COVER_FEATURES, *SMAP_FEATURES,
    *VEGETATION_FEATURES, *GFS_FEATURES, *EXTRA_WEATHER_3D[2], *EXTRA_WEATHER_3D[3], "fwi",
]

environmental = grid_table.copy()
for table in (
    weather_day1, weather_days_2_3.drop(columns="gfs_run_utc"),
    smap_features, ndmi_features, fwi_features,
):
    environmental = environmental.merge(table, on=KEYS, how="left", validate="one_to_one")
for table in (terrain, land_cover):
    environmental = environmental.merge(table, on=CELL_KEYS, how="left", validate="many_to_one")

if len(environmental) != len(grid_table):
    raise ValueError("The final table does not match the expected issue and cell count.")
if np.isinf(environmental[ENVIRONMENT_FEATURES].to_numpy(dtype=float)).any():
    raise ValueError("The final table contains infinite values.")

coverage = environmental[ENVIRONMENT_FEATURES].notna().groupby(environmental["year"]).mean()
coverage_path = OUTPUT_ROOT / "environmental_feature_coverage.csv"
coverage.to_csv(coverage_path, index_label="year")
display(coverage.round(4).T)

path = OUTPUT_ROOT / "environmental_features.parquet"
atomic_parquet(environmental, path)
print(f"Saved {len(environmental):,} rows to {path}.")
print(f"Annual feature coverage: {coverage_path}")